<a href="https://colab.research.google.com/github/Abhinaya-VS/EARLY-DETECTION-OF-ALZHEIMERS-DISEASE-USING-MULTI-MODAL-AND-EXPLAINABLE-AI/blob/main/ad_minor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from sklearn.utils.class_weight import compute_class_weight



In [ ]:
from google.colab import drive
drive.flush_and_unmount()  # safely unmount any previous mount


Drive not mounted, so nothing to flush and unmount.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
IMG_SIZE = (176, 208)
BATCH_SIZE = 32
NUM_CLASSES = 4
AUTOTUNE = tf.data.AUTOTUNE

TRAIN_DIR = "/content/drive/MyDrive/AD_Project/Combined Dataset/train"


TEST_DIR  = "/content/drive/MyDrive/AD_Project/Combined Dataset/test"
MODEL_PATH = "/content/drive/MyDrive/AD_Project/models/hybrid_resnet_merged.h5"


!mkdir -p /content/drive/MyDrive/AD_Project/models


In [ ]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=32,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

# ✅ GET class names IMMEDIATELY
class_names = train_ds.class_names
print(class_names)

# ✅ THEN optimize
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.cache().prefetch(tf.data.AUTOTUNE)


Found 10240 files belonging to 4 classes.
Found 1280 files belonging to 4 classes.
['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential

# Option 1: Using tf.keras.Sequential directly
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])


In [ ]:
def se_block(input_tensor, ratio=16):
    channels = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Dense(channels // ratio, activation="relu")(se)
    se = layers.Dense(channels, activation="sigmoid")(se)
    return layers.Multiply()([input_tensor, se])


In [ ]:
resnet = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(176, 208, 3)
)

vgg = tf.keras.applications.VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(176, 208, 3)
)

resnet.trainable = False
vgg.trainable = False


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
inputs = keras.Input(shape=(176, 208, 3))
x = data_augmentation(inputs)

x_res = tf.keras.applications.resnet50.preprocess_input(x)
x_vgg = tf.keras.applications.vgg16.preprocess_input(x)

res_feat = resnet(x_res, training=False)
vgg_feat = vgg(x_vgg, training=False)

# Attention applied
res_feat = se_block(res_feat)
vgg_feat = se_block(vgg_feat)

# Combine features
res_feat = layers.GlobalAveragePooling2D()(res_feat)
vgg_feat = layers.GlobalAveragePooling2D()(vgg_feat)

x = layers.concatenate([res_feat, vgg_feat])
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 176, 208,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 176, 208,  │          0 │ input_layer_2[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 176, 208)  │          0 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 176, 208)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 176, 208)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 176, 208)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 176, 208)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 176, 208)  │          0 │ sequential[0][0]  │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 176, 208,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 176, 208,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 176, 208,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 176, 208,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 6, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 5, 6, 512) │ 14,714,688 │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │    262,272 │ global_average_p

 Total params: 39,518,820 (150.75 MB)

 Trainable params: 1,216,420 (4.64 MB)

 Non-trainable params: 38,302,400 (146.11 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)



In [ ]:
for images, labels in train_ds.take(1):
    print(labels.shape)
    print(labels[0])
for _, y in train_ds.take(1):
    print("Train labels:", y.shape, y.dtype)

for _, y in test_ds.take(1):
    print("Test labels:", y.shape, y.dtype)


(32,)
tf.Tensor(0, shape=(), dtype=int32)
Train labels: (32,) <dtype: 'int32'>
Test labels: (32, 4) <dtype: 'float32'>


In [ ]:
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_cb = ModelCheckpoint(
    filepath=MODEL_PATH,      # Save final best model here
    monitor='val_accuracy',   # Monitor validation accuracy
    save_best_only=True,      # Save only when validation improves
    save_weights_only=False,  # Save full model (architecture + weights)
    verbose=1
)

# Optional: EarlyStopping (to avoid overfitting)
earlystop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Add these callbacks to your callbacks list

callbacks.append(checkpoint_cb)
callbacks.append(earlystop_cb)

# Fit your model
history_1 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

NameError: name 'callbacks' is not defined

In [ ]:
loss, acc, auc = model.evaluate(test_ds)
print("Accuracy:", acc)
print("AUC:", auc)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- Unfreeze top layers ---
for layer in resnet.layers[-40:]:
    layer.trainable = True

for layer in vgg.layers[-20:]:
    layer.trainable = True

# --- Recompile with low learning rate ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

# --- Checkpoint callback (save best model during fine-tuning) ---
checkpoint_cb = ModelCheckpoint(
    filepath= MODEL_PATH ,
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# Optional: EarlyStopping to prevent overfitting
earlystop_cb = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1
)

# Add to callbacks list
callbacks.append(checkpoint_cb)
callbacks.append(earlystop_cb)

# --- Resume training with fine-tuning ---
history_2 = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=30,             # total epochs for fine-tuning
    class_weight=class_weights,
    callbacks=callbacks
)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))



In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (176, 208)
NUM_CLASSES = 4

# ------------------
# Data augmentation
# ------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")

# ------------------
# SE Attention Block
# ------------------
def se_block(input_tensor, ratio=16):
    channels = input_tensor.shape[-1]
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Dense(channels // ratio, activation="relu")(se)
    se = layers.Dense(channels, activation="sigmoid")(se)
    return layers.Multiply()([input_tensor, se])

# ------------------
# Build Model (STEP 1)
# ------------------
def build_hybrid_model():

    inputs = layers.Input(shape=(*IMG_SIZE, 3), name="input")

    x = data_augmentation(inputs)

    # Preprocessing for each backbone
    x_res = tf.keras.applications.resnet50.preprocess_input(x)
    x_vgg = tf.keras.applications.vgg16.preprocess_input(x)

    # Backbones
    resnet = tf.keras.applications.ResNet50(
        include_top=False,
        weights="imagenet"
    )
    vgg = tf.keras.applications.VGG16(
        include_top=False,
        weights="imagenet"
    )

    resnet.trainable = False
    vgg.trainable = False

    res_feat = resnet(x_res, training=False)
    vgg_feat = vgg(x_vgg, training=False)

    # Attention
    res_feat = se_block(res_feat)
    vgg_feat = se_block(vgg_feat)

    # Pooling
    res_feat = layers.GlobalAveragePooling2D()(res_feat)
    vgg_feat = layers.GlobalAveragePooling2D()(vgg_feat)

    # Fusion
    x = layers.Concatenate()([res_feat, vgg_feat])

    # Classifier
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="Hybrid_ResNet_VGG_SE")
    return model


In [ ]:
model = build_hybrid_model()
model.load_weights(MODEL_PATH)
print("Weights loaded successfully ✅")


In [ ]:
model.save("/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras")


In [ ]:
model = tf.keras.models.load_model("improved_hybrid_resnet.keras")


In [ ]:
!ls /content/drive/MyDrive/AD_Project/models


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras"

model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully ✅")


In [ ]:
import tensorflow as tf
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)
loss, acc, auc = model.evaluate(test_ds, verbose=1)

print("Test Accuracy:", acc)
print("Test AUC:", auc)

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
MODEL_PATH = "/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras"
image_model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded ✅")


In [ ]:
model.get_layer("resnet50").summary()


In [ ]:


img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/Very Mild Impairment/1 (4).jpg"




In [ ]:
pip install lime


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from lime import lime_image
from skimage.segmentation import mark_boundaries


In [ ]:
import tensorflow as tf
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)
loss, acc, auc = model.evaluate(test_ds, verbose=1)

print("Test Accuracy:", acc)
print("Test AUC:", auc)

In [ ]:

img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/No Impairment/2 (48).jpg"
IMG_SIZE = (176, 208)  # <-- change
img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
print(class_names[np.argmax(pred)])



In [ ]:
!pip install lime



In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np
img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/Very Mild Impairment/5 (12).jpg"  # <-- change
img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
def predict_fn(images):
    images = images.astype(np.float32)
    return model.predict(images)
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    img_array[0].astype('double'),
    predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)
pred = model.predict(img_array)
confidence = np.max(pred)
print(class_names[np.argmax(pred)])
print("Confidence:", confidence)
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.axis('off')
plt.show()




In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np
img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/Moderate Impairment/17 (2).jpg"  # <-- change
img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
def predict_fn(images):
    images = images.astype(np.float32)
    return model.predict(images)
explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(
    img_array[0].astype('double'),
    predict_fn,
    top_labels=1,
    hide_color=0,
    num_samples=1000
)
temp, mask = explanation.get_image_and_mask(
    explanation.top_labels[0],
    positive_only=True,
    num_features=10,
    hide_rest=False
)
pred = model.predict(img_array)
confidence = np.max(pred)
print(class_names[np.argmax(pred)])
print("Confidence:", confidence)
plt.imshow(mark_boundaries(temp / 255.0, mask))
plt.axis('off')
plt.show()




In [ ]:
model.summary()


In [ ]:
# ==========================================
# 1. IMPORTS
# ==========================================
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


# ==========================================
# 2. IMAGE LOAD
# ==========================================
IMG_SIZE = (176, 208)

img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/Very Mild Impairment/18 (61).jpg"

img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


# ==========================================
# 3. CREATE GRAD MODEL
# ==========================================
last_conv_layer_name = "multiply_2"  # your layer

grad_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=[
        model.get_layer(last_conv_layer_name).output,
        model.output
    ]
)


# ==========================================
# 4. GRAD-CAM FUNCTION (FIXED)
# ==========================================
def make_gradcam_heatmap(img_array, grad_model):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        # ✅ Fix: ensure tensor
        if isinstance(predictions, list):
            predictions = predictions[0]

        # ✅ Get predicted class
        pred_index = tf.argmax(predictions[0])

        # ✅ Correct loss
        loss = predictions[0][pred_index]

    # Gradients
    grads = tape.gradient(loss, conv_outputs)

    # Pool gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Feature map
    conv_outputs = conv_outputs[0]

    # Heatmap
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # Normalize
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy(), int(pred_index), predictions[0][pred_index].numpy()


# ==========================================
# 5. GENERATE HEATMAP
# ==========================================
heatmap, pred_class, confidence = make_gradcam_heatmap(img_array, grad_model)

print("Predicted Class :", class_names[pred_class])
print("Confidence      :", f"{confidence*100:.2f} %")


# ==========================================
# 6. SUPERIMPOSE HEATMAP
# ==========================================
img = cv2.imread(img_path)
img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))

heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)


# ==========================================
# 7. DISPLAY
# ==========================================
plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.title("Grad-CAM Visualization")
plt.axis("off")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [ ]:
# Load dataset
df = pd.read_csv(
    "/content/drive/MyDrive/AD_Project/multimodel dataset/alzheimers_disease_data.csv"
)

# Drop non-numeric / unnecessary columns
df = df.drop(columns=["PatientID", "DoctorInCharge"])
X = df.drop(columns=["Diagnosis"])

In [ ]:
import joblib

feature_names = X.columns.tolist()

joblib.dump(
    feature_names,
    "/content/drive/MyDrive/AD_Project/models/clinical_feature_names.pkl"
)

print("✅ Feature names saved")

In [ ]:
FEATURE_COLUMNS = [
    "Age",
    "Gender",
    "EducationLevel",
    "BMI",
    "PhysicalActivity",
    "FamilyHistoryAlzheimers",
    "CardiovascularDisease",
    "Diabetes",
    "Hypertension",
    "MMSE",
    "FunctionalAssessment",
    "ADL",
    "MemoryComplaints"
]

X = df[FEATURE_COLUMNS].values
y = df["Diagnosis"].values  # 0 = No AD, 1 = AD


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
clinical_input = keras.Input(shape=(13,), name="clinical_input")

x = layers.Dense(64, activation="relu")(clinical_input)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(32, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(16, activation="relu", name="clinical_embedding")(x)

clinical_output = layers.Dense(
    2, activation="softmax", name="clinical_output"
)(x)

mlp_model = keras.Model(
    inputs=clinical_input,
    outputs=clinical_output,
    name="Clinical_MLP_Model"
)

mlp_model.summary()


In [ ]:
clinical_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)



In [ ]:
# ===============================
# 1. IMPORT LIBRARIES
# ===============================
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score

# ===============================
# 2. LOAD DATASET
# ===============================
df = pd.read_csv(
    "/content/drive/MyDrive/AD_Project/multimodel dataset/alzheimers_disease_data.csv"
)

# Drop unnecessary columns
df.drop(columns=["PatientID", "DoctorInCharge"], inplace=True)

# Separate features and target
X = df.drop(columns=["Diagnosis"])
y = df["Diagnosis"].values  # 0 or 1

# ===============================
# 3. TRAIN / TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ===============================
# 4. FEATURE SCALING
# ===============================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ===============================
# 5. ONE-HOT ENCODING LABELS
# ===============================
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=2)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=2)

# ===============================
# 6. CLASS WEIGHTS (IMPORTANT)
# ===============================
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

print("Class Weights:", class_weights)

# ===============================
# 7. BUILD MLP MODEL
# ===============================
clinical_input = layers.Input(shape=(X_train.shape[1],), name="clinical_input")

x = layers.Dense(64, activation="relu")(clinical_input)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(32, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(16, activation="relu", name="clinical_features")(x)

clinical_output = layers.Dense(2, activation="softmax", name="clinical_output")(x)

clinical_model = models.Model(
    inputs=clinical_input,
    outputs=clinical_output,
    name="Clinical_MLP_Model"
)

clinical_model.summary()

# ===============================
# 8. COMPILE MODEL
# ===============================
clinical_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

# ===============================
# 9. TRAIN MODEL
# ===============================
history = clinical_model.fit(
    X_train,
    y_train_cat,
    validation_data=(X_test, y_test_cat),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=6,
            restore_best_weights=True
        )
    ],
    verbose=1
)

# ===============================
# 10. EVALUATION
# ===============================
loss, acc = clinical_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"\nClinical Model Test Accuracy: {acc:.4f}")

# ===============================
# 11. SAVE MODEL + SCALER
# ===============================
clinical_model.save(
    "/content/drive/MyDrive/AD_Project/models/clinical_mlp_model.h5"
)

import joblib
joblib.dump(
    scaler,
    "/content/drive/MyDrive/AD_Project/models/clinical_scaler.pkl"
)

print("✅ Clinical model & scaler saved successfully")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))
print(class_weights)


In [ ]:
clinical_model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
    ]
)


In [ ]:
loss, acc = mlp_model.evaluate(X_test, y_test)
print(f"Clinical Model Accuracy: {acc:.4f}")


In [ ]:
# Save trained MLP
mlp_model.save(
    "/content/drive/MyDrive/AD_Project/models/clinical_mlp_final.h5"
)

# Save scaler
import joblib
joblib.dump(
    scaler,
    "/content/drive/MyDrive/AD_Project/models/clinical_scaler.save"
)

print("✅ Clinical MLP model & scaler saved successfully")


In [ ]:
import tensorflow as tf
import numpy as np

# Load trained models
model = tf.keras.models.load_model(
     "/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras"
)



print("Models loaded successfully")


In [ ]:
img_path ="/content/drive/MyDrive/AD_Project/Combined Dataset/test/Moderate Impairment/17 (2).jpg"

In [ ]:
import numpy as np
import tensorflow as tf

# -------------------------------
# STEP 1: Load & preprocess MRI image
# -------------------------------
IMG_SIZE = (176, 208)


   # CHANGE THIS

img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# -------------------------------
# STEP 2: MRI Model Prediction
# -------------------------------
mri_probs = model.predict(img_array)[0]

print("MRI Probabilities:")
for cls, prob in zip(class_names, mri_probs):
    print(f"{cls}: {prob*100:.2f}%")

predicted_index = np.argmax(mri_probs)
print("\nPredicted MRI Class:", class_names[predicted_index])

# Risk mapping for MRI (domain-based)
mri_risk_map = {
    "No Impairment": 0.1,
    "Very Mild Impairment": 0.4,
    "Mild Impairment": 0.7,
    "Moderate Impairment": 0.9
}

mri_risk = mri_risk_map[class_names[predicted_index]]

# -------------------------------
# STEP 3: Clinical Model Prediction
# -------------------------------
# Clinical input MUST be same preprocessing used during training
# Shape: (1, 13)

patient_dict = {
    "Age": 76,
    "Gender": 1,                     # Female
    "EducationLevel": 9,
    "BMI": 27.8,
    "PhysicalActivity": 1,           # Low activity
    "FamilyHistoryAlzheimers": 1,
    "CardiovascularDisease": 1,
    "Diabetes": 0,
    "Hypertension": 1,
    "MMSE": 18,                      # Cognitive impairment
    "FunctionalAssessment": 3.2,     # Reduced function
    "ADL": 2,                        # Needs assistance
    "MemoryComplaints": 1
}

# Create DataFrame
#patient_df = pd.DataFrame([patient_dict])

# Align columns EXACTLY like training
#patient_df = patient_df.reindex(columns=feature_names, fill_value=0)

# Apply scaler
#X_clinical_sample = scaler.transform(patient_df)
#clinical_prob = clinical_model.predict(X_clinical_sample)[0][1]


#decision = "Alzheimer's Disease" if final_risk >= 0.5 else "No Alzheimer's Disease"

#print("\n----- FINAL LATE FUSION RESULT -----")
#print("MRI Risk:", mri_risk)
#print("Clinical Risk:", clinical_risk)
#print("Final Risk Score:", final_risk)
#print("Final Decision:", decision)



In [ ]:
# ===============================
# 0. IMPORTS
# ===============================
import tensorflow as tf
import numpy as np
import pandas as pd
import joblib

# ===============================
# 1. LOAD MODELS
# ===============================
mri_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras",
    compile=False
)

clinical_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/AD_Project/models/clinical_mlp_model.h5",
    compile=False
)

# Load scaler and feature names
scaler = joblib.load(
    "/content/drive/MyDrive/AD_Project/models/clinical_scaler.pkl"
)

feature_names = joblib.load(
    "/content/drive/MyDrive/AD_Project/models/clinical_feature_names.pkl"
)

print("✅ Models and preprocessing files loaded")

# ===============================
# 2. MRI PREPROCESSING + PREDICTION
# ===============================
IMG_SIZE = (176, 208)



img = tf.keras.preprocessing.image.load_img(
    img_path,
    target_size=IMG_SIZE
)

img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

mri_probs = mri_model.predict(img_array)

# AD risk = 1 - No Impairment probability
NO_IMPAIRMENT_INDEX = 2   # change if your class order differs
mri_risk = float(1 - mri_probs[0][NO_IMPAIRMENT_INDEX])

# ===============================
# 3. CLINICAL PREPROCESSING + PREDICTION
# ===============================
patient_dict = {
    "Age": 76,
    "Gender": 1,                     # Female
    "EducationLevel": 9,
    "BMI": 27.8,
    "PhysicalActivity": 1,           # Low activity
    "FamilyHistoryAlzheimers": 1,
    "CardiovascularDisease": 1,
    "Diabetes": 0,
    "Hypertension": 1,
    "MMSE": 18,                      # Cognitive impairment
    "FunctionalAssessment": 3.2,     # Reduced function
    "ADL": 2,                        # Needs assistance
    "MemoryComplaints": 1
}
patient_df = pd.DataFrame([patient_dict])

# Enforce exact training feature order
patient_df = patient_df.reindex(columns=feature_names, fill_value=0)

X_clinical = scaler.transform(patient_df)

clinical_probs = clinical_model.predict(X_clinical)
clinical_risk = float(clinical_probs[0][1])  # AD probability





In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
IMG_SIZE = (176, 208)



img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
last_conv_layer_name = "multiply_2"  # ResNet SE output

grad_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=[
        model.get_layer(last_conv_layer_name).output,
        model.output
    ]
)
def make_gradcam_heatmap(img_array, grad_model):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy(), int(pred_index), predictions[0][pred_index].numpy()
heatmap, pred_class, confidence = make_gradcam_heatmap(img_array, grad_model)

#print("Predicted Class :", class_names[pred_class])
#print("Confidence      :", f"{confidence*100:.2f} %")
img = cv2.imread(img_path)
img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))

heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()



In [ ]:
from sklearn.metrics import accuracy_score

y_true = []
y_pred = []
y_score = []

X_clinical_test_np = np.array(X_test)
max_samples = len(X_clinical_test_np)

clinical_idx = 0

for img_batch, label_batch in test_ds:

    if clinical_idx >= max_samples:
        break

    batch_size = img_batch.shape[0]
    mri_probs = mri_model.predict(img_batch, verbose=0)
    mri_indices = np.argmax(mri_probs, axis=1)

    for i in range(batch_size):

        if clinical_idx >= max_samples:
            break

        # ---- MRI class ----
        mri_class = class_names[mri_indices[i]]

        # ---- Binary ground truth (FIX) ----
        true_binary = 0 if mri_class == "No Impairment" else 1

        # ---- MRI risk ----
        mri_risk = {
            "No Impairment": 0.1,
            "Very Mild Impairment": 0.4,
            "Mild Impairment": 0.7,
            "Moderate Impairment": 0.9
        }[mri_class]

        # ---- Clinical prediction ----
        clinical_sample = X_clinical_test_np[clinical_idx].reshape(1, -1)
        clinical_prob = clinical_model.predict(
            clinical_sample, verbose=0
        )[0][0]

        # ---- Late fusion ----
        final_risk = 0.6 * mri_risk + 0.4 * clinical_prob
        final_label = 1 if final_risk >= 0.5 else 0

        y_true.append(true_binary)
        y_pred.append(final_label)
        y_score.append(final_risk)

        clinical_idx += 1

# ---- Metrics ----
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_score = np.array(y_score)

accuracy = accuracy_score(y_true, y_pred)
print("Late Fusion Accuracy:", accuracy)


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_true, y_pred)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix – Late Fusion Model")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.colorbar()

classes = ["No AD", "AD"]
plt.xticks([0, 1], classes)
plt.yticks([0, 1], classes)

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()


In [ ]:
import tensorflow as tf
mri_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)
loss, acc, auc = mri_model.evaluate(test_ds, verbose=1)

print("Test Accuracy:", acc)
print("Test AUC:", auc)

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

y_true_mri = []
y_pred_mri = []

for images, labels in test_ds:
    preds = mri_model.predict(images, verbose=0)

    # Predictions → class index
    y_pred_mri.extend(np.argmax(preds, axis=1))

    # TRUE labels → FIX HERE
    if labels.ndim > 1:  # one-hot encoded
        y_true_mri.extend(np.argmax(labels.numpy(), axis=1))
    else:  # already sparse labels
        y_true_mri.extend(labels.numpy())


In [ ]:
y_true_mri = np.array(y_true_mri)
y_pred_mri = np.array(y_pred_mri)


In [ ]:
cm = confusion_matrix(y_true_mri, y_pred_mri)


In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(cm)
plt.title("Confusion Matrix – MRI Model")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.colorbar()

plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
from tensorflow.keras.utils import to_categorical

y_test_cat = to_categorical(y_test, num_classes=2)


In [ ]:
loss, acc = clinical_model.evaluate(X_test, y_test_cat, verbose=1)
print(f"Clinical Model Accuracy: {acc:.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predict classes
y_pred_clinical = np.argmax(
    clinical_model.predict(X_test, verbose=0),
    axis=1
)

y_true_clinical = np.argmax(y_test_cat, axis=1)

# Confusion Matrix
cm = confusion_matrix(y_true_clinical, y_pred_clinical)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No AD", "AD"]
)

plt.figure()
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – Clinical Model")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

mri_accuracy = 0.975781261920929        # ← replace with your MRI model accuracy
clinical_accuracy = acc   # from evaluate()

models = ["MRI Model", "Clinical Model"]
accuracies = [mri_accuracy, clinical_accuracy]

plt.figure()
plt.bar(models, accuracies)
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Accuracy Comparison: MRI vs Clinical Models")
plt.show()


In [ ]:
import pandas as pd

results_table = pd.DataFrame({
    "Model": ["MRI Model", "Clinical Model"],
    "Input Modality": ["MRI Images", "Clinical & Demographic Data"],
    "Accuracy": [mri_accuracy, clinical_accuracy],
    "Loss": ["-", loss],
    "Purpose": [
        "Structural brain pattern learning",
        "Risk factor and cognitive assessment"
    ]
})

results_table


In [ ]:
# ======================================
# 1. IMPORTS
# ======================================
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping


# ======================================
# 2. LOAD DATA
# ======================================
df = pd.read_csv(
    "/content/drive/MyDrive/AD_Project/multimodel dataset/alzheimers_disease_data.csv"
)


# ======================================
# 3. CLEAN DATA
# ======================================
drop_cols = ["PatientID", "DoctorInCharge"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])


# ======================================
# 4. ENCODE CATEGORICAL
# ======================================
df = pd.get_dummies(df, drop_first=True)


# ======================================
# 5. SPLIT FEATURES
# ======================================
X = df.drop("Diagnosis", axis=1)
y = df["Diagnosis"]


# ======================================
# 6. FEATURE SELECTION (IMPORTANT 🔥)
# ======================================
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X, y)

selector = SelectFromModel(rf, threshold="median")  # keep top 50%
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features)


# ======================================
# 7. TRAIN TEST SPLIT
# ======================================
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)


# ======================================
# 8. SCALING
# ======================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ======================================
# 9. CLASS WEIGHTS
# ======================================
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(weights))


# ======================================
# 🔥 10. ADVANCED MLP MODEL
# ======================================
model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])


# ======================================
# 11. COMPILE
# ======================================
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# ======================================
# 12. TRAIN
# ======================================
early_stop = EarlyStopping(patience=12, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=150,
    batch_size=16,
    validation_split=0.2,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)


# ======================================
# 13. EVALUATE
# ======================================
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


# ======================================
# 14. SAVE EVERYTHING
# ======================================
model.save("/content/drive/MyDrive/AD_Project/models/best_mlp_model.h5")
joblib.dump(scaler, "/content/drive/MyDrive/AD_Project/models/scaler.pkl")
joblib.dump(selected_features.tolist(), "/content/drive/MyDrive/AD_Project/models/features.pkl")


# ======================================
# 🔮 15. PREDICTION FUNCTION
# ======================================
from tensorflow.keras.models import load_model

def predict_patient(input_dict):
    model = load_model("/content/drive/MyDrive/AD_Project/models/best_mlp_model.h5")
    scaler = joblib.load("/content/drive/MyDrive/AD_Project/models/scaler.pkl")
    features = joblib.load("/content/drive/MyDrive/AD_Project/models/scaler.pkl")

    df_input = pd.DataFrame([input_dict])

    # One-hot encoding
    df_input = pd.get_dummies(df_input)

    # Align features
    df_input = df_input.reindex(columns=features, fill_value=0)

    # Scale
    df_scaled = scaler.transform(df_input)

    # Predict
    prob = model.predict(df_scaled)[0][0]
    pred = 1 if prob > 0.5 else 0

    return pred, prob

In [ ]:
!pip install -q tensorflow

In [ ]:
from tensorflow.keras.models import load_model, Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
def predict_patient(input_dict):
    model = load_model("/content/drive/MyDrive/AD_Project/models/best_mlp_model.h5")
    scaler = joblib.load("/content/drive/MyDrive/AD_Project/models/scaler.pkl")
    features = joblib.load("/content/drive/MyDrive/AD_Project/models/features.pkl")

    df_input = pd.DataFrame([input_dict])

    # One-hot encoding
    df_input = pd.get_dummies(df_input)

    # Align features
    df_input = df_input.reindex(columns=features, fill_value=0)

    # Scale
    df_scaled = scaler.transform(df_input)

    # Predict
    prob = model.predict(df_scaled)[0][0]
    pred = 1 if prob > 0.5 else 0

    return pred, prob

In [ ]:
# Sample input (you can modify values)
sample_input = {
    "Age": 60,
    "Gender": 1,
    "Ethnicity": 0,
    "Education": 4,
    "BMI": 22,
    "Smoking": 0,
    "Alcohol": 0,
    "PhysicalActivity": 4,
    "DietQuality": 5,
    "SleepQuality": 8,
    "FamilyHistoryAlzheimers": 0,
    "CardiovascularDisease": 0,
    "Diabetes": 0,
    "Depression": 0,
    "Hypertension": 0,
    "SystolicBP": 110,
    "DiastolicBP": 70,
    "CholesterolTotal": 160,
    "CholesterolHDL": 65,
    "CholesterolLDL": 85,
    "CognitiveTestScore": 29,
    "BehavioralProblems": 0,
    "ADL": 9,
    "MemoryComplaints": 0
}

# Call function
prediction, probability = predict_patient(sample_input)

# Print results
print("Prediction:", prediction)

if prediction == 1:
    print("Result: Alzheimer’s Detected")
else:
    print("Result: No Alzheimer’s")



In [ ]:
model = tf.keras.models.load_model(
     "/content/drive/MyDrive/AD_Project/models/improved_hybrid_resnet.keras"
)

In [ ]:
# ==========================================
# 1. IMPORTS
# ==========================================
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt


# ==========================================
# 2. IMAGE LOAD
# ==========================================
IMG_SIZE = (176, 208)

img_path = "/content/drive/MyDrive/AD_Project/Combined Dataset/test/No Impairment/2 (19).jpg"

img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


# ==========================================
# 3. CREATE GRAD MODEL
# ==========================================
last_conv_layer_name = "multiply_2"  # your layer

grad_model = tf.keras.models.Model(
    inputs=model.input,
    outputs=[
        model.get_layer(last_conv_layer_name).output,
        model.output
    ]
)


# ==========================================
# 4. GRAD-CAM FUNCTION (FIXED)
# ==========================================
def make_gradcam_heatmap(img_array, grad_model):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        # ✅ Fix: ensure tensor
        if isinstance(predictions, list):
            predictions = predictions[0]

        # ✅ Get predicted class
        pred_index = tf.argmax(predictions[0])

        # ✅ Correct loss
        loss = predictions[0][pred_index]

    # Gradients
    grads = tape.gradient(loss, conv_outputs)

    # Pool gradients
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Feature map
    conv_outputs = conv_outputs[0]

    # Heatmap
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # Normalize
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy(), int(pred_index), predictions[0][pred_index].numpy()


# ==========================================
# 5. GENERATE HEATMAP
# ==========================================
heatmap, pred_class, confidence = make_gradcam_heatmap(img_array, grad_model)

print("Predicted Class :", class_names[pred_class])
print("Confidence      :", f"{confidence*100:.2f} %")


# ==========================================
# 6. SUPERIMPOSE HEATMAP
# ==========================================
img = cv2.imread(img_path)
img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))

heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
heatmap = np.uint8(255 * heatmap)

heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

superimposed_img = cv2.addWeighted(img, 0.6, heatmap, 0.4, 0)


# ==========================================
# 7. DISPLAY
# ==========================================
plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB))
plt.title("Grad-CAM Visualization")
plt.axis("off")
plt.show()